In [0]:
# Célula 1 — setup
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

load_dotenv(os.path.join(os.getcwd(), ".env"), override=True)

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}
base_url = f"abfss://{os.getenv('ADLS_CONTAINER')}@{os.getenv('ADLS_STORAGE_ACCOUNT')}.dfs.core.windows.net"

TABELA_BRONZE_PRODUTOS = "workspace.squad1.bronze_rafael_bruna_ecommerce_produtos"
TABELA_BRONZE_CATEGORIAS = "workspace.squad1.bronze_rafael_bruna_ecommerce_categorias"

In [0]:
# Célula 2 — garante o schema, sempre qualificado com o catálogo
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.squad1")
spark.sql("SHOW TABLES IN workspace.squad1").show(100, truncate=False)

In [0]:
# Célula 3 — teste único: Delta gerenciado funciona mesmo no Serverless?
spark.createDataFrame([(1, "ok")], ["id", "msg"]) \
    .write.format("delta").mode("overwrite").saveAsTable("workspace.squad1.rafael_bruna_teste_delta")
print(spark.table("workspace.squad1.rafael_bruna_teste_delta").collect())

In [0]:
# Célula 4 — ingestão Bronze, sem transformação, idempotente por arquivo
def ingerir_bronze(nome_arquivo, tabela_bronze):
    df_bruto = (
        spark.read.format("parquet").options(**adls_options)
        .load(f"{base_url}/real-time-data/*/*/*/*/{nome_arquivo}.parquet")
        .withColumn("bronze_source_file", F.col("_metadata.file_path"))
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("_ano", F.year("bronze_ingested_at"))
        .withColumn("_mes", F.month("bronze_ingested_at"))
        .withColumn("_dia", F.dayofmonth("bronze_ingested_at"))
        .withColumn("_hora", F.hour("bronze_ingested_at"))
    )

    if spark.catalog.tableExists(tabela_bronze):
        ja_ingeridos = spark.table(tabela_bronze).select("bronze_source_file").distinct()
        df_novos = df_bruto.join(ja_ingeridos, "bronze_source_file", "left_anti")
    else:
        df_novos = df_bruto

    n = df_novos.count()
    if n == 0:
        print(f"{tabela_bronze}: nenhum arquivo novo")
        return

    (df_novos.write.format("delta").mode("append")
        .partitionBy("_ano", "_mes", "_dia", "_hora")
        .option("mergeSchema", "true")
        .saveAsTable(tabela_bronze))
    print(f"{tabela_bronze}: {n} linhas ingeridas")

ingerir_bronze("ecommerce_produtos", TABELA_BRONZE_PRODUTOS)
ingerir_bronze("ecommerce_categorias", TABELA_BRONZE_CATEGORIAS)